<a href="https://colab.research.google.com/github/yianchen903/my-project/blob/main/DG_Anderson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from dolfin import *
from fenics import *
import numpy as np
import sympy as sp
import math

def vecnorm(vec):
    return pow(inner(vec, vec), 0.5)

class ExcludedRegion(SubDomain):
    def inside(self, x, on_boundary):
        return (x[0] <= 0.5 and x[1] <= 0.5 )

x, y = sp.symbols('x[0] x[1]')

itersize0 = []
errsize0 = []
mingradsize0 = []

errwindow = [] # g(ek) error
iterwindow = [] # iter
exwindow = [] # ek error
extildewindow = [] # tlide ek error
thetawindow = [] # theta_k

class PdgSolver:
    def __init__(self, p, pdeg, size, penalty):
        self.p = p
        self.pdeg = pdeg
        self.size = size
        self.penalty = penalty


        self.mesh = None
        self.V = None
        self.W = None
        self.uex = None
        self.f = None
        self.bc = None


    def setup_problem(self):
      # Test1
        self.mesh = RectangleMesh(Point(1, 1), Point(2, 2), self.size, self.size)
        r = Expression("pow((pow(x[0],2)+pow(x[1],2)),0.5)", degree=5)


        self.uex = Expression(" pow(r,(p-2)/(p-1)) ", degree=5, r=r, p=self.p)
        qex1 = Expression(" ((p-2)/(p-1))*pow(r,-p/(p-1))*x[0] ", degree=5, r=r, p=self.p)
        qex2 = Expression(" ((p-2)/(p-1))*pow(r,-p/(p-1))*x[1] ", degree=5, r=r, p=self.p)
        self.Lex = as_vector((qex1,qex2))
        Coe = Constant(1.0)
        sigmaex1 = Expression(" pow((p-2)/(p-1),p-1)*pow(r,-2)*x[0] ", degree=5, r=r,p=self.p)
        sigmaex2 = Expression(" pow((p-2)/(p-1),p-1)*pow(r,-2)*x[1] ", degree=5, r=r,p=self.p)
        self.Aex = as_vector((sigmaex1,sigmaex2))
        f = 0
        self.f = Expression(sp.printing.ccode(f), degree=5)



        # Function spaces
        self.V = FunctionSpace(self.mesh, "DG", self.pdeg)
        self.W = VectorFunctionSpace(self.mesh, "DG", self.pdeg)

        # Dirichlet BC
        self.bc = DirichletBC(self.V, self.uex, "on_boundary")

    def _linear_init(self):

        r = Expression("pow((pow(x[0],2)+pow(x[1],2)),0.5)", degree=5)
        uex = Expression(" pow(r,(p-2)/(p-1)) ", degree=5, r=r,p=2)
        qex1 = Expression(" ((p-2)/(p-1))*pow(r,-p/(p-1))*x[0] ", degree=5, r=r,p=2)
        qex2 = Expression(" ((p-2)/(p-1))*pow(r,-p/(p-1))*x[1] ", degree=5, r=r,p=2)
        Lex = as_vector((qex1,qex2))
        sigmaex1 = Expression(" pow((p-2)/(p-1),p-1)*pow(r,-2)*x[0] ", degree=5, r=r,p=2)
        sigmaex2 = Expression(" pow((p-2)/(p-1),p-1)*pow(r,-2)*x[1] ", degree=5, r=r,p=2)
        Aex = as_vector((sigmaex1,sigmaex2))
        f = 0
        f = Expression(sp.printing.ccode(f), degree=5)


        u = TrialFunction(self.V)
        v = TestFunction(self.V)

        n = FacetNormal(self.mesh)
        h = CellDiameter(self.mesh)
        h_avg = (h('+') + h('-')) / 2

        alpha = Constant(self.penalty * self.pdeg * self.pdeg)
        sigma_b = alpha / h
        sigma_o = alpha / h_avg

        # DG bilinear form
        b0 = -inner(avg(grad(u)), jump(v, n)) * dS - inner(avg(grad(v)), jump(u, n)) * dS \
             + sigma_o * inner(jump(u, n), jump(v, n)) * dS

        b1 = -inner(grad(u), v * n) * ds - inner(grad(v), (u - uex) * n) * ds \
             + sigma_b * inner(v * n, (u - uex) * n) * ds

        N = inner(grad(u), grad(v)) * dx
        L = f * v * dx


        Lhs = N + b0 + b1
        Rhs = L

        form = Lhs-Rhs
        Lhs = lhs(form)
        Rhs = rhs(form)

        up = Function(self.V)
        solve(Lhs == Rhs, up, self.bc)

        return up

    def _solve_once(self, u0):

        u = TrialFunction(self.V)
        v = TestFunction(self.V)

        n = FacetNormal(self.mesh)
        h = CellDiameter(self.mesh)
        h_avg = (h('+') + h('-')) / 2

        delta = Constant(1e-10)
        alpha = Constant(self.penalty * self.pdeg * self.pdeg)
        sigma_b = alpha / h
        sigma_o = alpha / h_avg

        gradu0 = Function(self.W)
        gradu0 = project(grad(u0), self.W)

        # min_(x,y) mu(grad u) , mu(1/h ())
        mu_of_gradu0 = project(inner(gradu0,gradu0),self.V)
        mu_of_gradu0_array =  mu_of_gradu0.vector().get_local()

        count = np.sum(np.abs(mu_of_gradu0_array) < 1e-5)
        print("Number of elements with abs(mu) < 1e-5:", count)


        Nonlinear1 = delta + pow(vecnorm(gradu0), self.p - 2)
        Nonlinear2 = delta + pow(vecnorm(gradu0), self.p - 2)
        Nonlinear3 = delta + pow((1/h_avg) * vecnorm(jump(u0, n)), self.p - 2)
        Nonlinear4 = delta + pow(vecnorm(gradu0), self.p - 2)
        Nonlinear5 = delta + pow((1/h) * vecnorm((u0 - project(self.uex, self.V))), self.p - 2)

        b0 = -inner(avg(Nonlinear2 * grad(u)), jump(v, n)) * dS \
             - inner(Nonlinear3 * avg(grad(v)), jump(u, n)) * dS \
             + sigma_o * inner(jump(u, n), jump(v, n)) * dS

        b1 = -inner(Nonlinear4 * grad(u), v * n) * ds \
             - inner(Nonlinear5 * grad(v), (u - self.uex) * n) * ds \
             + sigma_b * inner(v * n, (u - self.uex) * n) * ds

        N = inner(Nonlinear1 * grad(u), grad(v)) * dx
        L = self.f * v * dx

        Lhs = N + b0 + b1
        Rhs = L

        form = Lhs-Rhs
        Lhs = lhs(form)
        Rhs = rhs(form)

        u_new = Function(self.V)
        solve(Lhs == Rhs, u_new, self.bc)

        return u_new, mu_of_gradu0_array

    def solve(self): # Picard

        iteriter0 = []
        erriter0 = []
        mingraditer0 = []

        u0 = self._linear_init()
        eps = 1.0
        iter = 0
        tol = 1.0e-12
        maxiter = 100
        while eps > tol and iter < maxiter:
            u_new, mu_of_gradu0_array_iter = self._solve_once(u0)
            eps = sqrt(assemble(inner(project(u_new - u0, self.V),
                                      project(u_new - u0, self.V)) * dx))
            print(' ')
            print(f"Iter {iter}: eps = {eps:.6e}")
            u0 = u_new
            iter += 1

            iteriter0.append(iter)
            erriter0.append(eps)
            mingraditer0.append(np.min(np.abs(mu_of_gradu0_array_iter)))
            # print("min grad u",mingraditer0)

        mingradsize0.append(mingraditer0)
        itersize0.append(iteriter0)
        errsize0.append(erriter0)


    def solve_Anderson(self,window,beta):

        u0 = self._linear_init()
        u0 = project(u0, self.V)

        G = []
        X = []


        h = CellDiameter(self.mesh)
        h_avg = (h('+') + h('-')) / 2
        alpha = Constant(self.penalty*self.pdeg*self.pdeg)
        sigma_b = alpha/h
        sigma_o = alpha/h_avg

        eps = 1.0
        tol = 1.0e-12
        iter = 1

        maxiter = 100

        exiter = [] # \tlide{e_{k+1}}
        extilde = [] # e_k
        erriter = [] # w_{k+1}
        iteriter = [] # iter
        thetaiter = []

        u1 = Function(self.V)  # u1
        u1 = self._solve_once(u0)[0] # tlide u1 = fu0 if iter=k=0
        u2 = Function(self.V) # u2
        u2 = self._solve_once(u1)[0]  # tlide u2

        u_er = project(u1-u0, self.V)
        err0  = sqrt(assemble(inner(u_er,u_er)*dx))
        utlide_er = project(u2-u1, self.V)
        etlide_er = sqrt(assemble(inner(utlide_er,utlide_er)*dx))

        extilde.append(etlide_er)
        exiter.append(err0)
        erriter.append(err0)
        iteriter.append(0)
        thetaiter.append(1.0)

        #
        # lowbd = 1 # 3-phase window lower bound
        # upbd = 6  # 3-phase window upper bound
        # m = window # window >= 1
        m = window
        beta = beta

        while eps > tol and iter < maxiter:

          u2 = Function(self.V) # u2
          k = iter
          mk = min(m,k)

          fu0, mu_of_gradu0_array_iter = self._solve_once(u0)  # solves IPDG fu0
          fu1, mu_of_gradu1_array_iter = self._solve_once(u1)  # solves IPDG fu1

          fu0_vec = fu0.vector().get_local() # np.array(fu0)
          u0_vec = u0.vector().get_local()   # np.array(u0)
          fu1_vec = fu1.vector().get_local() # np.array(fu1)
          u1_vec = u1.vector().get_local()   # np.array(u1)

          gu0 = fu0_vec - u0_vec     #  residual r0, f(u0)-u0
          gu1 = fu1_vec - u1_vec     #  residual r1, f(u1)-u1
          delta_g = gu1 - gu0        #  column of G  gu1 - gu0
          delta_u = u1_vec - u0_vec  #  column of X  u1 - u0


          # adds column
          delta_g = delta_g.tolist()
          delta_u = delta_u.tolist()
          G.append(delta_g)
          X.append(delta_u)

          # 3-phase window
          # examine_residul = np.linalg.norm(gu1, ord=2)
          # mk_examine = int(abs(np.ceil(-np.log10(examine_residul))))
          # if mk_examine <= lowbd:
          #   mk = lowbd # min(lowbd,k)
          # elif mk_examine >= upbd:
          #   mk = upbd # min(upbd,k)
          # else:
          #   mk = mk_examine # min(mk_examine,k)



          # removes column
          n = len(X)
          if n > mk:
            G = G[-mk: ]
            X = X[-mk: ]
          else:
            G = G[:]
            X = X[:]

          # calculates u_k
          Gk = np.array(G).T    # as column of G
          Xk = np.array(X).T    # as column of X


          # solves argmin||g_k - G_k*gamma||, gamma = (Gk^T Gk)^-1 Gk^T g,
          gamma = np.dot(np.dot(np.linalg.pinv(np.dot(Gk.T,Gk)) , Gk.T) , gu1)
          u2.vector().set_local(beta * ((u1_vec + gu1) - np.dot((Xk + Gk), gamma)) + (1.0-beta) * (u1_vec - np.dot(Xk,gamma))   )

          # print("gamma =",gamma)

          # theta_k
          alpha_k = np.zeros(len(gamma))
          wk_alpha = gamma[0]*gu0 + (1-gamma[0])*gu1
          theta_k =  np.linalg.norm(wk_alpha)/np.linalg.norm(gu1)
          thetaiter.append(theta_k)
          # print("theta_k =",theta_k)

          # min_(x,y) \mu(\grad u)
          mu_of_gradu2 = project(inner(grad(u2),grad(u2)),self.V)
          mu_of_gradu2_array =  mu_of_gradu2.vector().get_local()
          # print("min mu:", np.min(mu_of_gradu2_array))

          fu2, mu_of_gradu2_array_iter = self._solve_once(u2)  # solves DG fu2, fL2, fA2
          print("min mu:", np.min(mu_of_gradu2_array_iter))

          # iter error
          utlide_er = project(fu2-fu1, self.V)
          etliderr0 = sqrt(assemble(inner(utlide_er,utlide_er)*dx))
          u_ek = project(u2-u1, self.V)
          u_L2ek = sqrt(assemble(inner(u_ek,u_ek)*dx))
          u_er = project(u2-fu2, self.V)
          u_L2iter = sqrt(assemble(inner(u_er,u_er)*dx))
          u_exer = project(u2-self.uex, self.V)
          u_L2exiter = sqrt(assemble(inner(u_exer,u_exer)*dx))
          eps = u_L2iter

          # exiter.append(u_L2ek)
          # extilde.append(etliderr0)
          # erriter.append(eps)
          # iteriter.append(iter)

          # print("window = %d " %mk)
          print ("iter=%d: norm=%g norm=%g" %(iter, eps, u_L2exiter))
          print(" ")

          u0 = u1
          u1 = u2
          iter = iter + 1

        # extildewindow.append(extilde)
        # exwindow.append(exiter)
        # errwindow.append(erriter)
        # iterwindow.append(iteriter)
        # thetawindow.append(thetaiter)


        return u0

    def evaluate_error(self, uh):

        delta = 1e-10
        h = CellDiameter(self.mesh)
        h_avg = (h('+') + h('-')) / 2
        alpha = Constant(self.penalty * self.pdeg * self.pdeg)
        sigma_b = alpha / h
        sigma_o = alpha / h_avg

        u = project(uh,self.V)
        L = project(grad(uh),self.W)
        A = project(pow(delta + vecnorm(grad(uh)),self.p-2)*grad(uh),self.W)


        self.uex = project(self.uex, self.V)
        Ler = project(L-self.Lex, self.W)
        Aer = project(A-self.Aex, self.W)
        uer = project(u-self.uex, self.V)
        graduer = project(grad(u-self.uex),self.W)

        L_L2er = sqrt( assemble(inner(Ler,Ler)*dx) )
        A_L2er = sqrt( assemble(inner(Aer,Aer)*dx) )
        u_L2er = sqrt( assemble(uer*uer*dx) )
        u_BH1er = sqrt( assemble(inner(graduer,graduer)*dx)+ assemble(sigma_b*uer*uer*ds) + assemble(sigma_o*inner(jump(self.uex-u),jump(self.uex-u))*dS))

        print ("\n")
        print("p = %d " %self.p)
        print("alpha = %d " %self.penalty)
        print("u : degree %d , L : degree %d, A : degree %d" %(self.pdeg,self.pdeg,self.pdeg))
        print("mesh = %d x %d x 2" %(self.size,self.size))
        print ("|u-uh|_{L2} \t |L-Lh|_{L2} \t |A-Ah|_{L2} \t |u-uh|_{bH1}")
        print ("%.16f \t  %.16f \t  %.16f \t  %.16f" % (u_L2er, L_L2er, A_L2er, u_BH1er))
        print ("\n")




solver = PdgSolver(p=4.0, pdeg=1, size=40, penalty=10)
solver.setup_problem()
# uh = solver.solve() # Picard
uh = solver.solve_Anderson(window=1,beta=0.5)
solver.evaluate_error(uh)